# =============================================================================
# PROJECT 33: NAIVE BAYES WITH TF-IDF FOR SMS SPAM DETECTION
# By: Omar Sameh Mahmoud 25017251 , Ali Elsayed Ali 2400070
# =============================================================================
# Objectives:
# - Implement Multinomial Naive Bayes with TF-IDF vectorization
# - Compare against classical ML methods (SVM, Decision Tree, KNN)
# - Handle class imbalance, perform ablation studies, error analysis
# =============================================================================

In [4]:
# Cell 1: Environment Setup & Dependencies
# Installs required packages and imports all necessary libraries.

!pip install nltk scikit-learn pandas matplotlib seaborn -q

import pandas as pd
import numpy as np
import re
import os
import matplotlib.pyplot as plt
import seaborn as sns
import nltk

from nltk.corpus import stopwords
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, classification_report

# Set global random seed for full reproducibility
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

# Download NLTK datasets required for text preprocessing
nltk.download('stopwords', quiet=True)
nltk.download('punkt', quiet=True)

print("✅ Setup complete: All dependencies installed and imported successfully.")

✅ Setup complete: All dependencies installed and imported successfully.


In [5]:
# Cell 2: Dataset Download and Loading
# Downloads the UCI SMS Spam Collection dataset using Python libraries.

import urllib.request
import zipfile
import os
import pandas as pd

# Download the dataset using urllib (works on all platforms)
url = "https://archive.ics.uci.edu/ml/machine-learning-databases/00228/smsspamcollection.zip"
filename = "smsspamcollection.zip"

if not os.path.exists(filename):
    print("Downloading dataset...")
    urllib.request.urlretrieve(url, filename)
    print("Download complete.")
else:
    print("Dataset already downloaded.")

# Extract the zip file using zipfile module
if not os.path.exists("SMSSpamCollection"):
    print("Extracting files...")
    with zipfile.ZipFile(filename, 'r') as zip_ref:
        zip_ref.extractall(".")
    print("Extraction complete.")
else:
    print("Dataset already extracted.")

# Load dataset
df = pd.read_csv('SMSSpamCollection', sep='\t', names=['label', 'message'])
df['label'] = df['label'].map({'ham': 0, 'spam': 1})

print(f"Dataset loaded: {df.shape[0]} messages, {df.shape[1]} columns")
print(f"Class distribution:\n{df['label'].value_counts()}")
print(f"Spam ratio: {df['label'].mean()*100:.2f}%")

Download complete.
Extracting files...
Extraction complete.
Dataset loaded: 5572 messages, 2 columns
Class distribution:
label
0    4825
1     747
Name: count, dtype: int64
Spam ratio: 13.41%


In [8]:
# Cell 3: Text Preprocessing Pipeline
# Applies standard NLP cleaning: lowercase, punctuation/number removal, stopword filtering.
# Ensures consistent input format for feature extraction.

stop_words = set(stopwords.words('english'))

def preprocess_text(text):
    text = text.lower()
    text = re.sub(r'[^a-z\s]', ' ', text)  # Keep only alphabetic characters
    tokens = text.split()
    tokens = [t for t in tokens if t not in stop_words and len(t) > 2]
    return ' '.join(tokens)

df['clean'] = df['message'].apply(preprocess_text)
print("✅ Text preprocessing complete.")

✅ Text preprocessing complete.


In [9]:
# Cell 4: Train-Test Split & Feature Extraction
# Uses stratified sampling to preserve class imbalance in both sets.
# Implements TF-IDF (primary method) and Bag-of-Words (baseline comparison).

X_train, X_test, y_train, y_test = train_test_split(
    df['clean'], df['label'], test_size=0.2, random_state=RANDOM_STATE, stratify=df['label']
)
print(f"📈 Train size: {len(X_train)}, Test size: {len(X_test)}")

# TF-IDF Vectorization (Recommended per project methodology)
tfidf = TfidfVectorizer(max_features=3000, sublinear_tf=True)
X_train_tfidf = tfidf.fit_transform(X_train)
X_test_tfidf = tfidf.transform(X_test)
print(f"🔹 TF-IDF features: {X_train_tfidf.shape[1]}")

# Bag-of-Words Baseline
bow = CountVectorizer(max_features=3000)
X_train_bow = bow.fit_transform(X_train)
X_test_bow = bow.transform(X_test)
print(f"🔹 BoW features: {X_train_bow.shape[1]}")

📈 Train size: 4457, Test size: 1115
🔹 TF-IDF features: 3000
🔹 BoW features: 3000
